# 🌫️ 서울 PM2.5 데이터 분석 실습
### 안양대학교 일반대학원 미세먼지관리전공

---

**이 노트북에서 배우는 것:**
1. 📥 PM2.5 측정 데이터 불러오기 & 정제
2. 📊 계절별 농도 패턴 시각화
3. 🌡️ 기상인자와 PM2.5 상관관계 분석
4. 🗺️ 측정소별 농도 지도 시각화
5. 🤖 Random Forest로 고농도 예측

> **데이터:** 이 실습에서는 AirKorea 형식의 샘플 데이터를 사용합니다.  
> 실제 연구에서는 [AirKorea 공공데이터](https://www.airkorea.or.kr) 에서 다운로드하세요.

---
## 0️⃣ 패키지 설치 & 임포트

Colab에는 pandas, matplotlib 등이 기본 설치되어 있습니다.  
지도 시각화용 `folium`만 추가 설치합니다.

In [ ]:
# folium 설치 (지도 시각화용)
!pip install folium -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import folium
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정 (Colab용)
!apt-get install -y fonts-nanum -q
import matplotlib.font_manager as fm
fm._load_fontmanager(try_read_cache=False)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

print('✅ 모든 패키지 준비 완료!')

---
## 1️⃣ 데이터 불러오기 & 정제

### AirKorea 데이터 구조
실제 AirKorea CSV는 이런 형태입니다:
```
날짜, 측정소명, 시도, PM10, PM2.5, O3, NO2, CO, SO2
2023-01-01, 중구, 서울, 45, 28, 0.012, 0.034, 0.5, 0.003
```

**결측값 처리가 중요합니다:**
- AirKorea는 결측값을 `-999`로 표기
- 장비 오류, 점검 기간 등으로 발생
- 분석 전 반드시 제거해야 합니다

In [ ]:
# ── 샘플 데이터 생성 (AirKorea 형식) ──────────────────────────────────────
# 실제 수업에서는 AirKorea에서 받은 CSV를 read_csv()로 불러오면 됩니다

np.random.seed(42)
n = 1460  # 4년치 일별 데이터

dates = pd.date_range('2020-01-01', periods=n, freq='D')

# 계절별 PM2.5 패턴 반영 (봄/겨울 高, 여름 低)
def seasonal_pm25(dates):
    values = []
    for d in dates:
        month = d.month
        if month in [12, 1, 2]:   # 겨울 — 난방 + 기류 정체
            base = 35
        elif month in [3, 4, 5]:  # 봄 — 황사 + 중국 영향
            base = 40
        elif month in [6, 7, 8]:  # 여름 — 강수 세정 효과
            base = 18
        else:                      # 가을
            base = 25
        values.append(max(0, np.random.normal(base, 12)))
    return values

pm25_values = seasonal_pm25(dates)

# 결측값 5% 포함 (-999로 표기 — AirKorea 실제 형식)
missing_idx = np.random.choice(n, size=int(n * 0.05), replace=False)
for i in missing_idx:
    pm25_values[i] = -999

# 기상 데이터 (PM2.5와 상관관계 반영)
temp = np.random.normal(13, 10, n) + np.sin(np.arange(n) * 2 * np.pi / 365) * 12
humidity = np.random.normal(60, 15, n)
wind_speed = np.abs(np.random.normal(2.5, 1.2, n))

df_raw = pd.DataFrame({
    '날짜': dates,
    '측정소': np.random.choice(['중구', '강남구', '노원구', '영등포구', '마포구'], n),
    'PM25': pm25_values,
    'PM10': [v * 1.6 + np.random.normal(0, 5) if v != -999 else -999 for v in pm25_values],
    '기온': temp,
    '습도': humidity,
    '풍속': wind_speed,
})

print(f'📋 원본 데이터: {len(df_raw)}행 × {len(df_raw.columns)}열')
print(f'❌ 결측값(-999) 포함 행: {(df_raw["PM25"] == -999).sum()}개')
df_raw.head()

In [ ]:
# ── 데이터 정제 ────────────────────────────────────────────────────────────

df = df_raw.copy()

# ① -999 → NaN 변환 후 제거
df.replace(-999, np.nan, inplace=True)
df.dropna(subset=['PM25'], inplace=True)

# ② 날짜 인덱스 설정
df['날짜'] = pd.to_datetime(df['날짜'])
df.set_index('날짜', inplace=True)

# ③ 계절 컬럼 추가
def get_season(month):
    if month in [3, 4, 5]:  return '봄'
    elif month in [6, 7, 8]:  return '여름'
    elif month in [9, 10, 11]: return '가을'
    else: return '겨울'

df['계절'] = df.index.month.map(get_season)
df['월'] = df.index.month
df['연도'] = df.index.year

# ④ WHO 기준 초과 여부
df['WHO초과'] = df['PM25'] > 15   # WHO 연평균 기준 15μg/m³
df['나쁨'] = df['PM25'] > 35      # 국내 '나쁨' 기준 35μg/m³

print(f'✅ 정제 후 데이터: {len(df)}행')
print(f'📊 PM2.5 기초 통계:')
print(df['PM25'].describe().round(1))
print(f'\n⚠️  WHO 기준(15μg/m³) 초과 비율: {df["WHO초과"].mean()*100:.1f}%')
print(f'🔴 "나쁨" 기준(35μg/m³) 초과 비율: {df["나쁨"].mean()*100:.1f}%')

---
## 2️⃣ 계절별 농도 패턴 시각화

### 왜 계절별로 다를까?
| 계절 | 주요 원인 |
|------|----------|
| 봄 | 황사, 중국 대륙 고기압 확장 |
| 여름 | 강수에 의한 세정 효과, 남풍 계열 |
| 가을 | 이동성 고기압 → 대기 안정 → 축적 |
| 겨울 | 난방 연료 연소, 기온역전층 형성 |

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('서울 PM2.5 농도 계절별 분석 (2020–2023)', fontsize=15, fontweight='bold', y=1.01)

season_order = ['봄', '여름', '가을', '겨울']
palette = {'봄': '#FF9800', '여름': '#4CAF50', '가을': '#FF5722', '겨울': '#2196F3'}

# ① 계절별 박스플롯
ax1 = axes[0, 0]
season_data = [df[df['계절'] == s]['PM25'].values for s in season_order]
bp = ax1.boxplot(season_data, labels=season_order, patch_artist=True,
                  medianprops={'color': 'white', 'linewidth': 2})
for patch, season in zip(bp['boxes'], season_order):
    patch.set_facecolor(palette[season])
    patch.set_alpha(0.8)
ax1.axhline(y=15, color='blue', linestyle='--', linewidth=1.5, label='WHO 기준 (15)')
ax1.axhline(y=35, color='red', linestyle='--', linewidth=1.5, label='나쁨 기준 (35)')
ax1.set_ylabel('PM2.5 (μg/m³)')
ax1.set_title('계절별 PM2.5 분포')
ax1.legend(fontsize=9)
ax1.grid(axis='y', alpha=0.3)

# ② 월별 평균
ax2 = axes[0, 1]
monthly_mean = df.groupby('월')['PM25'].mean()
colors = ['#FF9800','#FF9800','#4CAF50','#FF9800','#FF9800','#4CAF50',
          '#4CAF50','#4CAF50','#FF5722','#FF5722','#FF5722','#2196F3']
bars = ax2.bar(monthly_mean.index, monthly_mean.values, color=colors, alpha=0.85, edgecolor='white')
ax2.axhline(y=15, color='blue', linestyle='--', linewidth=1.5, label='WHO 기준')
ax2.axhline(y=35, color='red', linestyle='--', linewidth=1.5, label='나쁨 기준')
for bar, val in zip(bars, monthly_mean.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.0f}', ha='center', va='bottom', fontsize=8)
ax2.set_xticks(range(1, 13))
ax2.set_xticklabels(['1월','2월','3월','4월','5월','6월','7월','8월','9월','10월','11월','12월'], fontsize=8)
ax2.set_ylabel('PM2.5 평균 (μg/m³)')
ax2.set_title('월별 PM2.5 평균 농도')
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.3)

# ③ 연도별 "나쁨" 일수
ax3 = axes[1, 0]
bad_days = df[df['나쁨']].groupby('연도').size()
ax3.bar(bad_days.index, bad_days.values, color='#E53935', alpha=0.85, edgecolor='white', width=0.5)
for i, (yr, cnt) in enumerate(bad_days.items()):
    ax3.text(yr, cnt + 0.5, f'{cnt}일', ha='center', fontsize=10, fontweight='bold')
ax3.set_ylabel('일수')
ax3.set_title('연도별 PM2.5 "나쁨" 초과 일수 (>35μg/m³)')
ax3.set_xticks(bad_days.index)
ax3.grid(axis='y', alpha=0.3)

# ④ 30일 이동평균 시계열
ax4 = axes[1, 1]
daily = df.groupby(df.index)['PM25'].mean()
rolling = daily.rolling(window=30, center=True).mean()
ax4.fill_between(daily.index, daily.values, alpha=0.15, color='gray')
ax4.plot(rolling.index, rolling.values, color='#1976D2', linewidth=2, label='30일 이동평균')
ax4.axhline(y=15, color='blue', linestyle='--', linewidth=1, alpha=0.7, label='WHO 기준')
ax4.axhline(y=35, color='red', linestyle='--', linewidth=1, alpha=0.7, label='나쁨 기준')
ax4.set_ylabel('PM2.5 (μg/m³)')
ax4.set_title('PM2.5 시계열 (30일 이동평균)')
ax4.legend(fontsize=9)
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('pm25_seasonal_analysis.png', bbox_inches='tight', dpi=150)
plt.show()
print('💾 그래프 저장: pm25_seasonal_analysis.png')

In [ ]:
# ── 계절별 기초통계 요약표 ─────────────────────────────────────────────────
summary = df.groupby('계절')['PM25'].agg(
    평균='mean', 중앙값='median', 표준편차='std',
    최솟값='min', 최댓값='max'
).round(1).reindex(season_order)

summary['나쁨_비율(%)'] = (
    df.groupby('계절')['나쁨'].mean() * 100
).reindex(season_order).round(1)

print('📋 계절별 PM2.5 기초 통계 (μg/m³)')
print('=' * 58)
print(summary.to_string())
print('\n💡 논문 결과 문단 예시:')
spring_mean = summary.loc['봄', '평균']
summer_mean = summary.loc['여름', '평균']
print(f'"봄철 PM2.5 평균 농도는 {spring_mean}μg/m³으로 여름철({summer_mean}μg/m³)에 비해'
      f' {spring_mean - summer_mean:.1f}μg/m³ 높게 나타났다."')

---
## 3️⃣ 기상인자와 PM2.5 상관관계 분석

**대기확산모델링 이론 / 파이썬 기반 환경 통계분석 수업 연계**

PM2.5 농도는 기상 조건에 크게 영향을 받습니다:
- **풍속** ↑ → PM2.5 ↓ (확산 효과)
- **습도** ↑ → PM2.5 ↑ (흡습 성장, 겨울 난방 연소)
- **기온** ↑ → PM2.5 ↓ (대기 불안정 → 혼합층 발달)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('PM2.5와 기상인자 상관관계', fontsize=14, fontweight='bold')

meteo_vars = [
    ('기온', '기온 (°C)', '#FF7043'),
    ('습도', '상대습도 (%)', '#42A5F5'),
    ('풍속', '풍속 (m/s)', '#66BB6A'),
]

for ax, (var, xlabel, color) in zip(axes, meteo_vars):
    sample = df.sample(500, random_state=42)  # 점 너무 많으면 느려서 샘플링
    ax.scatter(sample[var], sample['PM25'], alpha=0.3, s=15, color=color)

    # 추세선
    z = np.polyfit(sample[var], sample['PM25'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(sample[var].min(), sample[var].max(), 100)
    ax.plot(x_line, p(x_line), color='black', linewidth=2, linestyle='--')

    corr = df[var].corr(df['PM25'])
    ax.set_xlabel(xlabel)
    ax.set_ylabel('PM2.5 (μg/m³)')
    ax.set_title(f'PM2.5 vs {var}\n상관계수 r = {corr:.3f}')
    ax.axhline(y=35, color='red', linestyle=':', alpha=0.5)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('pm25_meteo_correlation.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── 상관계수 히트맵 ────────────────────────────────────────────────────────
corr_vars = ['PM25', 'PM10', '기온', '습도', '풍속']
corr_matrix = df[corr_vars].corr().round(3)

fig, ax = plt.subplots(figsize=(7, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)  # 상삼각 마스크
sns.heatmap(
    corr_matrix, annot=True, fmt='.3f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, ax=ax,
    cbar_kws={'shrink': 0.8}
)
ax.set_title('PM2.5 & 기상인자 상관계수 행렬', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('pm25_correlation_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()

print('\n📊 PM2.5와의 상관계수:')
print(corr_matrix['PM25'].drop('PM25').sort_values(key=abs, ascending=False))
print('\n💡 해석: 절댓값이 클수록 상관관계 강함')
print('   |r| > 0.5: 강한 상관 / 0.3~0.5: 중간 / < 0.3: 약한 상관')

---
## 4️⃣ 측정소별 PM2.5 농도 지도 시각화

**SMART 측정 및 분석 실습 수업 연계**

`folium`으로 서울 측정소별 PM2.5 평균 농도를 인터랙티브 지도에 표시합니다.

In [ ]:
# 서울 측정소 좌표 (실제 AirKorea 측정소 위치)
station_coords = {
    '중구':    (37.5640, 126.9749),
    '강남구':  (37.5172, 127.0473),
    '노원구':  (37.6541, 127.0564),
    '영등포구':(37.5264, 126.8962),
    '마포구':  (37.5544, 126.9240),
}

station_pm25 = df.groupby('측정소')['PM25'].agg(['mean', 'std', 'count']).round(1)
station_pm25.columns = ['평균', '표준편차', '데이터수']

# Folium 지도 생성
m = folium.Map(
    location=[37.5665, 126.9780],
    zoom_start=11,
    tiles='CartoDB positron'
)

# 농도별 색상 함수
def pm25_color(value):
    if value < 15:   return '#00E400'  # 좋음 (파란색)
    elif value < 35: return '#FFFF00'  # 보통 (노란색)
    elif value < 75: return '#FF7E00'  # 나쁨 (주황색)
    else:            return '#FF0000'  # 매우나쁨 (빨간색)

for station, row in station_pm25.iterrows():
    if station in station_coords:
        lat, lon = station_coords[station]
        pm_val = row['평균']
        color = pm25_color(pm_val)

        # 원형 마커
        folium.CircleMarker(
            location=[lat, lon],
            radius=pm_val / 3,
            color='gray', weight=1,
            fill=True, fill_color=color, fill_opacity=0.8,
            popup=folium.Popup(
                f'<b>{station}</b><br>'
                f'평균 PM2.5: <b>{pm_val}μg/m³</b><br>'
                f'표준편차: {row["표준편차"]}μg/m³<br>'
                f'데이터 수: {row["데이터수"]}개',
                max_width=200
            ),
            tooltip=f'{station}: {pm_val}μg/m³'
        ).add_to(m)

        folium.Marker(
            location=[lat + 0.003, lon],
            icon=folium.DivIcon(
                html=f'<div style="font-size:11px;font-weight:bold;color:#333">{station}<br>{pm_val}μg/m³</div>',
                icon_size=(80, 30)
            )
        ).add_to(m)

# 범례 추가
legend_html = '''
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
     background:white;padding:12px;border-radius:8px;
     border:2px solid #ccc;font-size:13px;">
<b>PM2.5 등급</b><br>
<span style="color:#00E400">●</span> 좋음 (&lt;15μg/m³)<br>
<span style="color:#CCCC00">●</span> 보통 (15–35μg/m³)<br>
<span style="color:#FF7E00">●</span> 나쁨 (35–75μg/m³)<br>
<span style="color:#FF0000">●</span> 매우나쁨 (&gt;75μg/m³)
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

print('📋 측정소별 PM2.5 평균 농도:')
print(station_pm25)
print('\n🗺️ 지도 생성 완료! 마커 클릭하면 상세 정보를 볼 수 있습니다.')
m

---
## 5️⃣ Random Forest로 PM2.5 "나쁨" 예측

**AI기반 미세먼지 예측 및 관리 수업 연계**

오늘의 기상 조건(기온, 습도, 풍속)과 전날 PM2.5를 입력으로,  
내일 PM2.5가 **"나쁨"(35μg/m³ 초과) 여부를 예측**합니다.

**머신러닝 분류 모델 흐름:**
```
입력(X): 기온, 습도, 풍속, 전날PM2.5, 월
      ↓
Random Forest 모델
      ↓
출력(y): 내일 나쁨(1) / 나쁨아님(0)
```

In [ ]:
# ── 특성(Feature) 준비 ────────────────────────────────────────────────────
df_ml = df.copy().sort_index()

# 전날 PM2.5를 특성으로 추가 (lag feature)
df_ml['전날PM25'] = df_ml['PM25'].shift(1)
df_ml['전날나쁨'] = (df_ml['전날PM25'] > 35).astype(int)
df_ml.dropna(inplace=True)

features = ['기온', '습도', '풍속', '전날PM25', '월']
target = '나쁨'

X = df_ml[features]
y = df_ml[target].astype(int)

# 학습/테스트 분리 (시계열이라 shuffle=False)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

print(f'🔢 학습 데이터: {len(X_train)}개 / 테스트 데이터: {len(X_test)}개')
print(f'📊 나쁨 비율 — 학습: {y_train.mean()*100:.1f}% / 테스트: {y_test.mean()*100:.1f}%')

In [ ]:
# ── 모델 학습 & 평가 ──────────────────────────────────────────────────────
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    class_weight='balanced'  # 불균형 클래스 처리
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = (y_pred == y_test).mean()

print(f'✅ 모델 정확도: {accuracy*100:.1f}%\n')
print('📋 분류 성능 리포트:')
print(classification_report(y_test, y_pred,
                              target_names=['나쁨 아님', '나쁨'],
                              digits=3))

In [ ]:
# ── 결과 시각화 ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Random Forest PM2.5 "나쁨" 예측 결과', fontsize=13, fontweight='bold')

# ① 혼동행렬
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['예측: 나쁨아님', '예측: 나쁨'],
            yticklabels=['실제: 나쁨아님', '실제: 나쁨'],
            ax=axes[0], cbar=False)
axes[0].set_title('혼동 행렬 (Confusion Matrix)')
axes[0].set_ylabel('실제 값')
axes[0].set_xlabel('예측 값')

# ② 특성 중요도
importances = pd.Series(model.feature_importances_, index=features)
importances = importances.sort_values(ascending=True)
colors = ['#EF5350' if i == importances.idxmax() else '#90CAF9'
          for i in importances.index]
bars = axes[1].barh(importances.index, importances.values,
                     color=colors, edgecolor='white')
for bar, val in zip(bars, importances.values):
    axes[1].text(val + 0.005, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=10)
axes[1].set_xlabel('특성 중요도 (Feature Importance)')
axes[1].set_title('예측에 중요한 변수')
axes[1].set_xlim(0, importances.max() * 1.25)
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('pm25_rf_result.png', bbox_inches='tight', dpi=150)
plt.show()

top_feature = importances.idxmax()
print(f'\n🏆 가장 중요한 예측 변수: "{top_feature}"')
print('💡 전날 PM2.5 농도가 가장 강력한 예측 인자 → 대기오염의 연속성(관성) 반영')

---
## 6️⃣ 내 연구에 적용하기

### 실제 AirKorea 데이터로 분석하려면

```python
# ① AirKorea에서 CSV 다운로드
# https://www.airkorea.or.kr → 에어코리아 데이터 → 측정소별 데이터 조회

# ② Colab에 업로드
from google.colab import files
uploaded = files.upload()  # 파일 선택 창 열림

# ③ 불러오기
import pandas as pd
df_real = pd.read_csv('airkorea_2023.csv', encoding='euc-kr')  # 한글 인코딩 주의

# ④ 이 노트북의 코드를 그대로 적용!
```

### 논문에서 이 분석을 어떻게 쓸까?

| 노트북 섹션 | 논문 위치 |
|------------|----------|
| 1. 데이터 정제 | 3장 연구방법 — 데이터 수집 및 전처리 |
| 2. 계절별 시각화 | 4장 결과 — PM2.5 시공간 분포 특성 |
| 3. 상관관계 분석 | 4장 결과 — 기상인자와의 관계 |
| 4. 지도 시각화 | 4장 결과 — 공간적 분포 |
| 5. 예측 모델 | 4장 결과 — AI 기반 예측 성능 |

---
*안양대학교 일반대학원 환경공학과 미세먼지관리전공 실습 자료*  
*github.com/kwangorithm/finddust*